# PROJECT STAGE

Current stage: Research exploration

Code may later be migrated to src modules once stabilized.

# 1. Notebook Metadata

In [ ]:
# ============================================================
# NOTEBOOK METADATA
# ============================================================

NOTEBOOK_NAME = "00_download_market_data"
AUTHOR = "Juan Manuel Giacame"
CREATED = "2026-03-13"
UPDATED = "2026-03-13"
PROJECT = "macro-market-analysis"
STAGE = "Data Ingestion"
VERSION = "0.1.0"
GIT_HASH = ""  # completar con commit hash si se usa git
EXPERIMENT_ID = f"{NOTEBOOK_NAME}_{CREATED}_{VERSION}"

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
PURPOSE = """
Download raw market data for all assets in the defined universe
using yfinance. 

Data includes OHLCV, dividends, and stock splits.

The resulting raw data is stored per asset in:

data/raw/{asset}.parquet
"""

# ------------------------------------------------------------
# INPUT DATASETS
# ------------------------------------------------------------
INPUT_DATASETS = [
    "Universe defined in src/quant_research/data/registry/universe_registry.py"
]

# ------------------------------------------------------------
# OUTPUT DATASETS
# ------------------------------------------------------------
OUTPUT_DATASETS = [
    "data/raw/{asset}.parquet"
]

# ------------------------------------------------------------
# DATASET
# ------------------------------------------------------------
DATASET = "Raw Market Data"

# ------------------------------------------------------------
# DATASET DESCRIPTION
# ------------------------------------------------------------
DATASET_DESCRIPTION = """
Raw market data provides the foundational input for all 
downstream processing, including calculation of features such as
momentum, volatility, and systemic metrics.

Columns include:
    - Open
    - High
    - Low
    - Close
    - Adj Close
    - Volume
    - Dividends
    - Stock Splits
"""

# ------------------------------------------------------------
# ASSETS UNIVERSE
# ------------------------------------------------------------
ASSETS_UNIVERSE = "All assets defined in universe_registry.py (configurable)"

# ------------------------------------------------------------
# DEPENDENCIES
# ------------------------------------------------------------
DEPENDENCIES = ["pandas >= 2.0", "numpy", "yfinance >= 0.2", "pathlib", "pyarrow"]

# ------------------------------------------------------------
# NOTES
# ------------------------------------------------------------
NOTES = """
Data is downloaded with:
    - auto_adjust=False
    - actions=True

Incremental download logic ensures only new data is fetched.
Future pipeline steps include processing and normalization
in 01_data_pipeline.ipynb.
"""

# 2. Imports

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

# ------------------------
# Standard library
# ------------------------
from pathlib import Path
from datetime import datetime, timedelta

# ------------------------
# Data manipulation
# ------------------------
import pandas as pd
import numpy as np

# ------------------------
# Data sources
# ------------------------
import yfinance as yf

# ------------------------
# Data storage / parquet
# ------------------------
import pyarrow  # required by pandas for parquet support

# ------------------------
# Visualization
# ------------------------
import plotly.express as px

# 3. Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

from pathlib import Path
from datetime import datetime, timedelta

# ------------------------------------------------------------
# DATA PATHS
# ------------------------------------------------------------
from quant_research.config.paths import RAW_DATA_PATH, PROCESSED_DATA_PATH
# ------------------------------------------------------------
# RAW DATA PARAMETERS
# ------------------------------------------------------------
AUTO_ADJUST = False   # keep raw OHLCV prices
ACTIONS = True        # include dividends and stock splits

# Download interval (yfinance supports: '1d', '1wk', '1mo', etc.)
INTERVAL = "1d"

# Default date range (can be overridden per asset)
START_DATE = "2000-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

# ============================================================
# RAW DATA SCHEMA
# ============================================================

EXPECTED_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close",
    "Adj Close",
    "Volume",
    "Dividends",
    "Stock Splits",
    "Capital Gains"
]

# ------------------------------------------------------------
# UNIVERSE CONFIGURATION
# ------------------------------------------------------------
# Universe of assets is defined in src/quant_research/data/registry/universe_registry.py
ASSETS_UNIVERSE_SOURCE = "universe_registry.py"

# Optional sample assets for quick testing or plots
SAMPLE_ASSETS_FOR_TEST = ["BTC", "SPY"]

# ------------------------------------------------------------
# DOWNLOAD OPTIONS
# ------------------------------------------------------------
OVERWRITE_EXISTING = False   # if True, re-download all data
INCREMENTAL_DOWNLOAD = True  # only fetch new dates

# ------------------------------------------------------------
# RANDOM SEED / REPRODUCIBILITY
# ------------------------------------------------------------
RANDOM_SEED = 42

# 4. Core Computation

In [ ]:
from quant_research.data.raw.download_pipeline import run_download_pipeline

run_download_pipeline()

In [ ]:
# ============================================================
# RAW DATA SANITY CHECK
# ============================================================

import pandas as pd

sample_assets = ["SPY", "BTC"]

for asset in sample_assets:

    path = RAW_DATA_PATH / f"{asset}.parquet"
    df = pd.read_parquet(path)

    print(f"\n{asset}")
    print("rows:", len(df))
    print("start:", df.index.min())
    print("end:", df.index.max())
    print("columns:", len(df))
    print("columns:", list(df.columns))

In [ ]:
# ============================================================
# RAW DATA REPAIR / NORMALIZATION
# ============================================================
from quant_research.data.raw.raw_data_helpers import (
    get_last_date,
    download_asset,
    validate_download,
    normalize_columns,
    merge_with_existing,
    save_parquet,
)

from quant_research.data.registry.universe_registry import ASSET_UNIVERSE
from quant_research.config.paths import RAW_DATA_PATH

import pandas as pd

print("Starting RAW data repair...\n")

for asset in ASSET_UNIVERSE:

    symbol = asset.symbol
    path = RAW_DATA_PATH / f"{symbol}.parquet"

    if not path.exists():
        print(f"{symbol}: file not found")
        continue

    df = pd.read_parquet(path)

    # --------------------------------------------------------
    # Flatten MultiIndex columns (yfinance issue)
    # --------------------------------------------------------

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # --------------------------------------------------------
    # Normalize schema
    # --------------------------------------------------------

    df = normalize_columns(df)

    # --------------------------------------------------------
    # Clean index
    # --------------------------------------------------------

    df.index = pd.to_datetime(df.index)

    df = df.sort_index()

    df = df[~df.index.duplicated(keep="last")]

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    df.to_parquet(path)

    print(f"{symbol}: repaired ({len(df)} rows)")

print("\nRAW data repair completed.")

# 5. Diagnostics/validation

In [ ]:
# ============================================================
# RAW DATA SANITY CHECK
# ============================================================

import pandas as pd

sample_assets = ["SPY", "BTC"]

for asset in sample_assets:

    path = RAW_DATA_PATH / f"{asset}.parquet"
    df = pd.read_parquet(path)

    print(f"\n{asset}")
    print("rows:", len(df))
    print("start:", df.index.min())
    print("end:", df.index.max())
    print("columns:", len(df))
    print("columns:", list(df.columns))

In [ ]:
# ============================================================
# OHLC CONSISTENCY CHECK
# ============================================================

import pandas as pd

assets = ["SPY", "GLD", "BTC"]

for symbol in assets:

    df = pd.read_parquet(RAW_DATA_PATH / f"{symbol}.parquet")

    bad_rows = df[
        (df["Low"] > df["Open"]) |
        (df["Low"] > df["Close"]) |
        (df["High"] < df["Open"]) |
        (df["High"] < df["Close"])
    ]

    print(f"\n{symbol}")
    print("bad rows:", len(bad_rows))

In [ ]:
# ============================================================
# EXTREME RETURN CHECK
# ============================================================

assets = ["SPY", "GLD", "BTC"]

for symbol in assets:

    df = pd.read_parquet(RAW_DATA_PATH / f"{symbol}.parquet")

    returns = df["Close"].pct_change()

    extreme = returns.abs() > 0.5

    print(f"\n{symbol}")
    print("extreme moves:", extreme.sum())

In [ ]:
# ============================================================
# DATE GAP CHECK
# ============================================================

assets = ["SPY", "GLD", "BTC"]

for symbol in assets:

    df = pd.read_parquet(RAW_DATA_PATH / f"{symbol}.parquet")

    gaps = df.index.to_series().diff()

    large_gaps = gaps > pd.Timedelta(days=5)

    print(f"\n{symbol}")
    print("large gaps:", large_gaps.sum())

# 6. Visualization

In [ ]:
# ============================================================
# PRICE SERIES VISUAL CHECK
# ============================================================

import plotly.graph_objects as go

sample_assets = ["SPY", "GLD", "BTC"]

fig = go.Figure()

for symbol in sample_assets:

    path = RAW_DATA_PATH / f"{symbol}.parquet"
    df = pd.read_parquet(path)

    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df["Close"],
            name=symbol,
            mode="lines"
        )
    )

fig.update_layout(
    title="Raw Close Prices (Sanity Check)",
    xaxis_title="Date",
    yaxis_title="Price",
    template="plotly_white",
    height=600
)

fig.show()

In [ ]:
# ============================================================
# DAILY RETURNS CHECK
# ============================================================

fig = go.Figure()

for symbol in sample_assets:

    path = RAW_DATA_PATH / f"{symbol}.parquet"
    df = pd.read_parquet(path)

    returns = df["Close"].pct_change()

    fig.add_trace(
        go.Scatter(
            x=returns.index,
            y=returns*100,
            name=symbol,
            mode="lines"
        )
    )

fig.update_layout(
    title="Daily Returns (Sanity Check)",
    xaxis_title="Date",
    yaxis_title="Return",
    template="plotly_white",
    height=600
)

fig.show()

In [ ]:
# ============================================================
# NORMALIZED VOLUME (BASE 100) - LOG SCALE
# ============================================================

import plotly.graph_objects as go

sample_assets = ["SPY", "GLD", "BTC"]

fig = go.Figure()

for symbol in sample_assets:

    path = RAW_DATA_PATH / f"{symbol}.parquet"
    df = pd.read_parquet(path)

    vol = df["Volume"]
    vol_norm = vol / vol.iloc[0] * 100

    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=vol_norm,
            name=symbol,
            mode="lines"
        )
    )

fig.update_layout(
    title="Normalized Trading Volume (Base=100, Log Scale)",
    xaxis_title="Date",
    yaxis_title="Normalized Volume",
    yaxis_type="log",
    template="plotly_white",
    height=600
)

fig.show()

In [ ]:
# ============================================================
# DOLLAR VOLUME
# ============================================================

fig = go.Figure()

for symbol in sample_assets:

    path = RAW_DATA_PATH / f"{symbol}.parquet"
    df = pd.read_parquet(path)

    dollar_vol = df["Close"] * df["Volume"]

    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=dollar_vol,
            name=symbol,
            mode="lines"
        )
    )

fig.update_layout(
    title="Dollar Trading Volume",
    xaxis_title="Date",
    yaxis_title="Dollar Volume",
    yaxis_type="log",
    template="plotly_white",
    height=600
)

fig.show()

In [ ]:
# ============================================================
# DOLLAR VOLUME ROLLING MEAN
# ============================================================

import plotly.graph_objects as go

sample_assets = ["SPY", "GLD", "BTC"]

fig = go.Figure()

for symbol in sample_assets:

    df = pd.read_parquet(RAW_DATA_PATH / f"{symbol}.parquet")

    dollar_vol = df["Close"] * df["Volume"]

    dv_roll = dollar_vol.rolling(30).mean()

    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=dv_roll,
            name=symbol
        )
    )

fig.update_layout(
    title="30D Rolling Dollar Volume",
    yaxis_type="log",
    template="plotly_white"
)

fig.show()

In [ ]:
# ============================================================
# NORMALIZED PRICE (BASE 100)
# ============================================================

import plotly.graph_objects as go

sample_assets = ["SPY", "GLD", "BTC"]

fig = go.Figure()

for symbol in sample_assets:

    path = RAW_DATA_PATH / f"{symbol}.parquet"
    df = pd.read_parquet(path)

    price = df["Close"]

    price_norm = price / price.iloc[0] * 100

    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=price_norm,
            name=symbol,
            mode="lines"
        )
    )

fig.update_layout(
    title="Normalized Price (Base = 100)",
    xaxis_title="Date",
    yaxis_title="Price (Normalized)",
    template="plotly_white",
    height=600
)

fig.show()